# 資料對齊：依 Subject ID 跨 Session 比對

目的：從各 session 資料夾掃描 `.mat` 檔，以 subject ID 為鍵，確保同一個人的三個 session 資料對齊，再建立正確的 fold 分配。

In [ ]:
import os
import numpy as np
import h5py

In [15]:

def get_subject_id(filename):
    """從檔名提取 subject ID，取 _ses- 前面的部分
    例：sub-21_ses-01_task-...mat  →  'sub-21'
    """
    return filename.split('_ses-')[0]

def scan_session_folder(folder_path):
    """掃描資料夾，回傳 {subject_id: filepath} 的 dict"""
    result = {}
    for fname in sorted(os.listdir(folder_path)):
        if fname.endswith('.mat'):
            sub_id = get_subject_id(fname)
            result[sub_id] = os.path.join(folder_path, fname)
    return result

def load_band_energy(filepath):
    """載入 MATLAB v7.3 (.mat HDF5) 檔案，回傳 band_energy (32, 39, 5)
    h5py 讀出來維度是反轉的 (5, 39, 32)，用 .T 轉回 (32, 39, 5)
    """
    with h5py.File(filepath, 'r') as f:
        data = np.array(f['band_energy']).T  # (5,39,32) → (32,39,5)
    return data

def find_complete_subjects(ss01_map, ss02_map, ss03_map):
    """回傳三個 session 都有的 subject ID（排序固定）"""
    return sorted(set(ss01_map) & set(ss02_map) & set(ss03_map))

def load_group(ss01_map, ss02_map, ss03_map, complete_subs):
    """依照 complete_subs 順序載入三個 session 的資料
    回傳 X_ss01, X_ss02, X_ss03，每個 shape: (n_subs, 32, 39, 5)
    """
    X1, X2, X3 = [], [], []
    for sub_id in complete_subs:
        X1.append(load_band_energy(ss01_map[sub_id]))
        X2.append(load_band_energy(ss02_map[sub_id]))
        X3.append(load_band_energy(ss03_map[sub_id]))
    return np.array(X1), np.array(X2), np.array(X3)


In [16]:
# ── 設定各 group 的資料夾路徑 ──
BASE_AMATEUR = r'D:\Tseng\圍棋\amateur\band_energy_batch_outputs_2026'
BASE_MASTER  = r'D:\Tseng\圍棋\master\band_energy_batch_outputs_2026'
BASE_PROF    = r'D:\Tseng\圍棋\prof\band_energy_batch_outputs_2026'

# 如果 master / prof 資料夾結構不同，請修改以下路徑
PATHS = {
    'amateur': {
        'ss01': os.path.join(BASE_AMATEUR, 'ss01'),
        'ss02': os.path.join(BASE_AMATEUR, 'ss02'),
        'ss03': os.path.join(BASE_AMATEUR, 'ss03'),
    },
    'master': {
        'ss01': os.path.join(BASE_MASTER, 'ss01'),
        'ss02': os.path.join(BASE_MASTER, 'ss02'),
        'ss03': os.path.join(BASE_MASTER, 'ss03'),
    },
    'prof': {
        'ss01': os.path.join(BASE_PROF, 'ss01'),
        'ss02': os.path.join(BASE_PROF, 'ss02'),
        'ss03': os.path.join(BASE_PROF, 'ss03'),
    },
}

In [17]:
# ── 掃描資料夾並確認每個 group 的 subject 完整性 ──
maps = {}
complete = {}

for group in ['amateur', 'master', 'prof']:
    m1 = scan_session_folder(PATHS[group]['ss01'])
    m2 = scan_session_folder(PATHS[group]['ss02'])
    m3 = scan_session_folder(PATHS[group]['ss03'])
    maps[group] = (m1, m2, m3)

    c = find_complete_subjects(m1, m2, m3)
    complete[group] = c

    missing_ss03 = sorted((set(m1) & set(m2)) - set(m3))
    missing_ss02 = sorted((set(m1) & set(m3)) - set(m2))
    missing_ss01 = sorted((set(m2) & set(m3)) - set(m1))

    print(f"[{group}]")
    print(f"  ss01={len(m1)}人  ss02={len(m2)}人  ss03={len(m3)}人")
    print(f"  三個 session 都齊全：{len(c)} 人")
    if missing_ss03: print(f"  缺 ss03：{missing_ss03}")
    if missing_ss02: print(f"  缺 ss02：{missing_ss02}")
    if missing_ss01: print(f"  缺 ss01：{missing_ss01}")
    print()

[amateur]
  ss01=30人  ss02=30人  ss03=28人
  三個 session 都齊全：28 人
  缺 ss03：['sub-21', 'sub-36']

[master]
  ss01=20人  ss02=21人  ss03=21人
  三個 session 都齊全：20 人
  缺 ss01：['sub-78']

[prof]
  ss01=8人  ss02=8人  ss03=8人
  三個 session 都齊全：8 人



In [18]:
# ── 載入資料（只保留三個 session 都齊全的 subject）──
print("載入 amateur...")
X_amateur_ss01, X_amateur_ss02, X_amateur_ss03 = load_group(*maps['amateur'], complete['amateur'])

print("載入 master...")
X_master_ss01, X_master_ss02, X_master_ss03 = load_group(*maps['master'], complete['master'])

print("載入 prof...")
X_prof_ss01, X_prof_ss02, X_prof_ss03 = load_group(*maps['prof'], complete['prof'])

print("\n完成！")
print(f"amateur: ss01={X_amateur_ss01.shape}, ss02={X_amateur_ss02.shape}, ss03={X_amateur_ss03.shape}")
print(f"master:  ss01={X_master_ss01.shape}, ss02={X_master_ss02.shape}, ss03={X_master_ss03.shape}")
print(f"prof:    ss01={X_prof_ss01.shape}, ss02={X_prof_ss02.shape}, ss03={X_prof_ss03.shape}")

載入 amateur...
載入 master...
載入 prof...

完成！
amateur: ss01=(28, 32, 39, 5), ss02=(28, 32, 39, 5), ss03=(28, 32, 39, 5)
master:  ss01=(20, 32, 39, 5), ss02=(20, 32, 39, 5), ss03=(20, 32, 39, 5)
prof:    ss01=(8, 32, 39, 5), ss02=(8, 32, 39, 5), ss03=(8, 32, 39, 5)


In [19]:

# ── 特徵提取 & Fold 分配 ──
N_FOLDS = 3
CLASS_NAMES = ['amateur', 'master', 'prof']

def extract_features(X):
    """X: (n_subs, 32ch, 39time, 5bands) → (n_subs, 160)
    log → 去前 3 個時間點 → mean over time → flatten(32ch × 5bands)
    """
    X = np.log(X + 1e-8)       # log transform
    X = X[:, :, 3:, :]         # 去前 3 時間點 → (n, 32, 36, 5)
    X = X.mean(axis=2)         # mean over time → (n, 32, 5)
    return X.reshape(len(X), -1)  # flatten    → (n, 160)

def make_fold_ids(n_subs):
    """subject_index % N_FOLDS，同一 subject 三個 session 都拿到同一 fold"""
    return np.arange(n_subs) % N_FOLDS

# 每個 group 的 subject 數
n_a = len(complete['amateur'])
n_m = len(complete['master'])
n_p = len(complete['prof'])
print(f"amateur={n_a}人  master={n_m}人  prof={n_p}人")

# 提取特徵
Fa1 = extract_features(X_amateur_ss01)
Fa2 = extract_features(X_amateur_ss02)
Fa3 = extract_features(X_amateur_ss03)

Fm1 = extract_features(X_master_ss01)
Fm2 = extract_features(X_master_ss02)
Fm3 = extract_features(X_master_ss03)

Fp1 = extract_features(X_prof_ss01)
Fp2 = extract_features(X_prof_ss02)
Fp3 = extract_features(X_prof_ss03)

# 按 class 順序疊合（與 fid_all 一致：amateur全session → master全session → prof全session）
X_all = np.vstack([Fa1, Fa2, Fa3,
                   Fm1, Fm2, Fm3,
                   Fp1, Fp2, Fp3])

y_all = np.concatenate([
    np.zeros(n_a * 3),
    np.ones(n_m * 3),
    np.full(n_p * 3, 2)
])

# fold ID：同一 subject index 在三個 session 都拿到同一 fold
fid_all = np.concatenate([
    make_fold_ids(n_a), make_fold_ids(n_a), make_fold_ids(n_a),  # amateur ss01, ss02, ss03
    make_fold_ids(n_m), make_fold_ids(n_m), make_fold_ids(n_m),  # master
    make_fold_ids(n_p), make_fold_ids(n_p), make_fold_ids(n_p),  # prof
])

print(f"\nX_all shape: {X_all.shape}  (應為 ({(n_a+n_m+n_p)*3}, 160))")
print(f"y 分佈: {np.bincount(y_all.astype(int)).tolist()}")
print("\nFold 分布:")
for k in range(N_FOLDS):
    mask = fid_all == k
    cnt  = np.bincount(y_all[mask].astype(int), minlength=3)
    print(f"  Fold {k}: total={mask.sum()}, amateur={cnt[0]} master={cnt[1]} prof={cnt[2]}")


amateur=28人  master=20人  prof=8人

X_all shape: (168, 160)  (應為 (168, 160))
y 分佈: [84, 60, 24]

Fold 分布:
  Fold 0: total=60, amateur=30 master=21 prof=9
  Fold 1: total=57, amateur=27 master=21 prof=9
  Fold 2: total=51, amateur=27 master=18 prof=6


In [20]:

# ── 列出每個 Fold 裡的 subject ID ──
print("=" * 60)
for k in range(N_FOLDS):
    fold_idx = np.where(np.arange(n_a) % N_FOLDS == k)[0]
    print(f"\nFold {k} 的 subject：")
    print(f"  amateur ({len(fold_idx)}人): {[complete['amateur'][i] for i in fold_idx]}")

    fold_idx = np.where(np.arange(n_m) % N_FOLDS == k)[0]
    print(f"  master  ({len(fold_idx)}人): {[complete['master'][i] for i in fold_idx]}")

    fold_idx = np.where(np.arange(n_p) % N_FOLDS == k)[0]
    print(f"  prof    ({len(fold_idx)}人): {[complete['prof'][i] for i in fold_idx]}")

# ── 確認沒有同一個人同時出現在不同 fold ──
print("\n" + "=" * 60)
print("交叉確認（同一個人不應出現在多個 fold）：")
for group in ['amateur', 'master', 'prof']:
    subs = complete[group]
    n = len(subs)
    fold_map = {subs[i]: i % N_FOLDS for i in range(n)}
    # 每個 subject 只對應一個 fold，不可能重複，但確認 subject list 本身沒有重名
    if len(set(subs)) == len(subs):
        print(f"  [{group}] OK：{n} 人，無重複 subject ID")
    else:
        from collections import Counter
        dup = [s for s, c in Counter(subs).items() if c > 1]
        print(f"  [{group}] 警告：有重複的 subject ID！{dup}")



Fold 0 的 subject：
  amateur (10人): ['sub-22', 'sub-33', 'sub-40', 'sub-43', 'sub-46', 'sub-49', 'sub-P026', 'sub-P029', 'sub-P032', 'sub-sub50']
  master  (7人): ['sub-52', 'sub-57', 'sub-60', 'sub-67', 'sub-70', 'sub-73', 'sub-77']
  prof    (3人): ['sub-53', 'sub-64', 'sub-74']

Fold 1 的 subject：
  amateur (9人): ['sub-23', 'sub-38', 'sub-41', 'sub-44', 'sub-47', 'sub-51', 'sub-P027', 'sub-P030', 'sub-sub34']
  master  (7人): ['sub-55', 'sub-58', 'sub-61', 'sub-68', 'sub-71', 'sub-75', 'sub-80']
  prof    (3人): ['sub-54', 'sub-65', 'sub-79']

Fold 2 的 subject：
  amateur (9人): ['sub-24', 'sub-39', 'sub-42', 'sub-45', 'sub-48', 'sub-P025', 'sub-P028', 'sub-P031', 'sub-sub35']
  master  (6人): ['sub-56', 'sub-59', 'sub-63', 'sub-69', 'sub-72', 'sub-76']
  prof    (2人): ['sub-62', 'sub-66']

交叉確認（同一個人不應出現在多個 fold）：
  [amateur] OK：28 人，無重複 subject ID
  [master] OK：20 人，無重複 subject ID
  [prof] OK：8 人，無重複 subject ID
